# 04 | Renda por pessoa (BEA)

A planilha do BEA tem anos como colunas. Vamos colocá-los como linhas, para combinar com as outras fontes.

**Como usar:** execute os blocos de código 1, 2 e 3, nessa ordem. Leia a explicação antes de cada bloco. Cada bloco reúne pequenas ações que fazem parte da mesma etapa.

## Bloco 1: Ler CSV

Use a exportação `SAINC1`, indicador `Per capita personal income (dollars)`, em `data/raw/bea/personal_income.csv`.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
import pandas as pd
arquivo = ROOT / "data/raw/bea/personal_income.csv"
bea = pd.read_csv(arquivo,skiprows=3,encoding="utf-8-sig",dtype={"GeoFIPS":str},low_memory=False)
anos = [str(a) for a in range(2019,2025)]
bea = bea[["GeoFIPS","GeoName"]+anos]
print("Tabela original:",bea.shape)

## Bloco 2: Organizar

`melt()` é o comando que transforma as colunas 2019, 2020 etc. em uma coluna de ano e outra de renda.

In [ ]:
longa = bea.melt(id_vars=["GeoFIPS","GeoName"],value_vars=anos,var_name="year",value_name="income_per_capita")
longa = longa.rename(columns={"GeoName":"state"})
longa["state"] = longa["state"].str.replace(r"\s*\*$","",regex=True).str.strip()
longa["year"] = longa["year"].astype(int)
longa["income_per_capita"] = pd.to_numeric(longa["income_per_capita"],errors="coerce")
cdc = pd.read_csv(ROOT / "data/raw/cdc_homicides_2019_2024.csv")
estados = set(cdc["name"]) - {"District of Columbia"}
limpo = longa[longa["state"].isin(estados)][["state","year","income_per_capita"]].copy()
display(limpo.head())

## Bloco 3: Conferir e salvar

O esperado são 300 linhas. A renda está em dólares **do próprio ano**, sem correção da inflação.

In [ ]:
assert limpo.shape == (300,3)
assert not limpo.duplicated(["state","year"]).any()
assert limpo["income_per_capita"].notna().all()
assert (limpo["income_per_capita"] > 0).all()
destino = ROOT / "data/processed/bea_income_2019_2024.csv"
destino.parent.mkdir(parents=True,exist_ok=True)
limpo.to_csv(destino,index=False)
print("OK, salvo:",destino.relative_to(ROOT))